# Getting Started

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## Connect to OpenRouter

OpenRouter is OpenAI-compatible — we use the `openai` Python library pointed at OpenRouter's endpoint.

In [ ]:
import os
from openai import OpenAI

# OpenRouter uses the OpenAI-compatible API
# Set OPENROUTER_API_KEY in your environment variables
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Model to use throughout this course
# OpenRouter format: "provider/model-name"
MODEL = "anthropic/claude-sonnet-4-5"


## Basic Example

Your first API call. Let's look at the raw response object first.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response)


Extract just the text content:

In [ ]:
print(response.choices[0].message.content)


## Roles

Chat LLMs support three roles in the conversation:

| Role | Purpose |
|---|---|
| `system` | Sets the model's persistent behavior — tone, format, restrictions |
| `user` | The message from the human |
| `assistant` | The model's previous replies (used in multi-turn conversations) |


In [ ]:
# Use system + user roles together
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer with the answer only, nothing else."},
        {"role": "user",   "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)


## The get_completion Helper

We wrap the API call into a helper function. We reuse this in every notebook.

In [ ]:
def get_completion(prompt, system="You are a helpful assistant.", temperature=0):
    """
    Simple wrapper around OpenRouter API.

    Args:
        prompt      : the user message
        system      : the system prompt
        temperature : 0 = deterministic, 1 = creative
    
    Returns:
        The model's response as a plain string.
    """
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ]
    )
    return response.choices[0].message.content


Test it:

In [ ]:
answer = get_completion("What is the capital of France?")
print(answer)


In [ ]:
# With a custom system prompt
answer = get_completion(
    prompt="What is the capital of France?",
    system="You are a geography teacher. Answer in exactly one word."
)
print(answer)


## Temperature

Temperature controls how random the output is.

- `temperature=0` → always the same answer ✅ use for NLP tasks  
- `temperature=1` → varied, creative answers — use for writing tasks


In [ ]:
question = "Give me a one-word description of Python the programming language."

for temp in [0.0, 0.5, 1.0]:
    answer = get_completion(question, temperature=temp)
    print(f"temperature={temp}  →  {answer}")


**Rule for this course:** Always use `temperature=0` for NLP tasks like  
classification, NER, and structured extraction — you need consistent, reproducible output.

## Switching Models

One of the benefits of OpenRouter is that you can switch models without changing any other code.


In [ ]:
# Compare the same question across two models
models = [
    "anthropic/claude-sonnet-4-5",
    "meta-llama/llama-3.1-8b-instruct",
]

for model_name in models:
    resp = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Explain NLP in one sentence."}]
    )
    print(f"Model: {model_name}")
    print(f"Answer: {resp.choices[0].message.content}")
    print()


You are now ready for **02_prompting_tips.ipynb**.